# 🚀 Fleeti — Générateur de Leads B2B

Génère automatiquement une liste de prospects qualifiés à partir des données publiques françaises.

**Résultat :** 2 fichiers téléchargeables
- `Prospects_Fleeti.xlsx` — tableau Excel lisible
- `Prospects_Fleeti_odoo.csv` — prêt à importer dans Odoo

---
**👉 Suis les 3 étapes ci-dessous**

In [ ]:
#@title ⚙️ Étape 1 — Installation (lancer une seule fois)
!pip install requests openpyxl -q
!wget -q https://raw.githubusercontent.com/LouiseFleeti/sales-academy-fleeti/main/prospection/prospection.py
print('✅ Prêt !')

In [ ]:
#@title 🎯 Étape 2 — Configure ta recherche

# ── TOKENS API ──────────────────────────────────────────
SIRENE_TOKEN = '4ab9e4a0-a10e-4627-8373-a9afd3a21332' #@param {type:"string"}
PAPPERS_TOKEN = '' #@param {type:"string"}

# ── SECTEUR ─────────────────────────────────────────────
SECTEUR = 'Transport routier' #@param ["Transport routier", "Transport frigorifique", "Messagerie / Logistique", "BTP - Gros oeuvre", "BTP - Second oeuvre", "BTP - Génie civil", "Ambulances / Transport sanitaire"]

# ── GÉOGRAPHIE ──────────────────────────────────────────
ZONE = 'France entière' #@param ["France entière", "Île-de-France", "Nouvelle-Aquitaine", "Occitanie", "Auvergne-Rhône-Alpes", "PACA", "Pays de la Loire", "Bretagne", "Hauts-de-France", "Grand Est", "Normandie", "Centre-Val de Loire", "Bourgogne-Franche-Comté", "Corse"]

# ── TAILLE ENTREPRISE ────────────────────────────────────
TAILLE = '10 à 499 salariés (PME)' #@param ["10 à 499 salariés (PME)", "200 à 999 salariés", "500+ salariés (Grands comptes)"]

# ── NOMBRE DE RÉSULTATS ──────────────────────────────────
NOMBRE = 100 #@param {type:"integer"}

print('✅ Configuration enregistrée')
print(f'   Secteur  : {SECTEUR}')
print(f'   Zone     : {ZONE}')
print(f'   Taille   : {TAILLE}')
print(f'   Résultats: {NOMBRE}')

In [ ]:
#@title 🚀 Étape 3 — Générer la liste et télécharger
import subprocess, os
from google.colab import files

# Mapping secteurs → codes APE
APE_MAP = {
    'Transport routier':               ['49.41A','49.41B','49.41C'],
    'Transport frigorifique':          ['49.41A','49.41B','49.41C','49.42Z'],
    'Messagerie / Logistique':         ['52.29A','52.29B','52.10B','49.41C'],
    'BTP - Gros oeuvre':               ['41.20A','41.20B','43.99C'],
    'BTP - Second oeuvre':             ['43.21A','43.22A','43.31Z','43.32A','43.34Z'],
    'BTP - Génie civil':               ['42.11Z','42.21Z','42.22Z','42.99Z'],
    'Ambulances / Transport sanitaire':['86.90A'],
}

# Mapping régions → départements
DEP_MAP = {
    'France entière':            [],
    'Île-de-France':             ['75','77','78','91','92','93','94','95'],
    'Nouvelle-Aquitaine':        ['16','17','19','23','24','33','40','47','64','79','86','87'],
    'Occitanie':                 ['09','11','12','30','31','32','34','46','48','65','66','81','82'],
    'Auvergne-Rhône-Alpes':      ['01','03','07','15','26','38','42','43','63','69','73','74'],
    'PACA':                      ['04','05','06','13','83','84'],
    'Pays de la Loire':          ['44','49','53','72','85'],
    'Bretagne':                  ['22','29','35','56'],
    'Hauts-de-France':           ['02','59','60','62','80'],
    'Grand Est':                 ['08','10','51','52','54','55','57','67','68','88'],
    'Normandie':                 ['14','27','50','61','76'],
    'Centre-Val de Loire':       ['18','28','36','37','41','45'],
    'Bourgogne-Franche-Comté':   ['21','25','39','58','70','71','89','90'],
    'Corse':                     ['2A','2B'],
}

# Mapping tailles → tranches effectifs
TAILLE_MAP = {
    '10 à 499 salariés (PME)':   ('11', '32'),
    '200 à 999 salariés':        ('31', '41'),
    '500+ salariés (Grands comptes)': ('41', '53'),
}

ape_codes = APE_MAP[SECTEUR]
departements = DEP_MAP[ZONE]
eff_min, eff_max = TAILLE_MAP[TAILLE]
output = '/content/Prospects_Fleeti.xlsx'

cmd = [
    'python3', 'prospection.py',
    '--sirene-token', SIRENE_TOKEN,
    '--ape', *ape_codes,
    '--effectif-min', eff_min,
    '--effectif-max', eff_max,
    '--limit', str(NOMBRE),
    '--output', output,
]
if PAPPERS_TOKEN:
    cmd += ['--pappers-token', PAPPERS_TOKEN]
else:
    cmd += ['--skip-pappers']
if departements:
    cmd += ['--departements', *departements]

print('⏳ Génération en cours...')
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print('ERREUR:', result.stderr)
else:
    csv_path = output.replace('.xlsx', '_odoo.csv')
    print('\n📥 Téléchargement des fichiers...')
    files.download(output)
    if os.path.exists(csv_path):
        files.download(csv_path)
    print('✅ Terminé !')